# Configuration


In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Load the dataset
accidents_df = pd.read_csv('../data/US_Accidents_March23_sampled_500k.csv')

# Data Preprocessing


In [3]:
# different feature groups we could investigate for clustering
WEATHER_COLUMNS = [
    "Temperature(F)", "Wind_Chill(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)", 
    "Wind_Speed(mph)", "Precipitation(in)", "Wind_Direction", "Weather_Condition"
]
TEMPORAL_COLUMNS = [
    "Start_Time", "End_Time", "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight",
    "Astronomical_Twilight"
]
GEOGRAPHIC_COLUMNS= [
    "Start_Lat", "Start_Lng", "End_Lat", "End_Lng", "City", "County", "State", 
    "Zipcode", "Country", "Timezone", "Airport_Code", "Street", "Distance(mi)"
]
INFRA_COLUMNS = [
    "Amenity", "Bump", "Crossing", "Give_Way", "Junction", "No_Exit", "Railway", 
    "Roundabout", "Station", "Stop", "Traffic_Calming", "Traffic_Signal"
]
# show what they like
display(accidents_df[WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS].info())
display(accidents_df[WEATHER_COLUMNS].head(2))
display(accidents_df[TEMPORAL_COLUMNS].head(2))
display(accidents_df[GEOGRAPHIC_COLUMNS].head(2))
display(accidents_df[INFRA_COLUMNS].head(2))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 40 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Temperature(F)         489534 non-null  float64
 1   Wind_Chill(F)          370983 non-null  float64
 2   Humidity(%)            488870 non-null  float64
 3   Pressure(in)           491072 non-null  float64
 4   Visibility(mi)         488709 non-null  float64
 5   Wind_Speed(mph)        463013 non-null  float64
 6   Precipitation(in)      357384 non-null  float64
 7   Wind_Direction         488803 non-null  object 
 8   Weather_Condition      488899 non-null  object 
 9   Start_Time             500000 non-null  object 
 10  End_Time               500000 non-null  object 
 11  Sunrise_Sunset         498517 non-null  object 
 12  Civil_Twilight         498517 non-null  object 
 13  Nautical_Twilight      498517 non-null  object 
 14  Astronomical_Twilight  498517 non-nu

None

,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Wind_Direction,Weather_Condition
0,77.0,77.0,62.0,29.92,10.0,5.0,0.0,NW,Fair
1,45.0,43.0,48.0,29.91,10.0,5.0,0.0,W,Fair


,Start_Time,End_Time,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,2019-06-12 10:10:56,2019-06-12 10:55:58,Day,Day,Day,Day
1,2022-12-03 23:37:14.000000000,2022-12-04 01:56:53.000000000,Night,Night,Night,Night


,Start_Lat,Start_Lng,End_Lat,End_Lng,City,County,State,Zipcode,Country,Timezone,Airport_Code,Street,Distance(mi)
0,30.641211,-91.153481,NaN,NaN,Zachary,East Baton Rouge,LA,70791-4610,US,US/Central,KBTR,Highway 19,0.000
1,38.990562,-77.399070,38.990037,-77.398282,Sterling,Loudoun,VA,20164-2813,US,US/Eastern,KIAD,Forest Ridge Dr,0.056


,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal
0,False,False,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,False,False


In [ ]:
# Exclude columns that are not realistic to impute but has substantial data loss
if "Wind_Chill(F)" in WEATHER_COLUMNS:
    WEATHER_COLUMNS.remove("Wind_Chill(F)")
for col in ["End_Lat", "End_Lng", "Zipcode", "Airport_Code", "Street"]:
    if col in GEOGRAPHIC_COLUMNS:
        GEOGRAPHIC_COLUMNS.remove(col)
display(accidents_df[WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS].info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500000 entries, 0 to 499999
Data columns (total 34 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Temperature(F)         489534 non-null  float64
 1   Humidity(%)            488870 non-null  float64
 2   Pressure(in)           491072 non-null  float64
 3   Visibility(mi)         488709 non-null  float64
 4   Wind_Speed(mph)        463013 non-null  float64
 5   Precipitation(in)      357384 non-null  float64
 6   Wind_Direction         488803 non-null  object 
 7   Weather_Condition      488899 non-null  object 
 8   Start_Time             500000 non-null  object 
 9   End_Time               500000 non-null  object 
 10  Sunrise_Sunset         498517 non-null  object 
 11  Civil_Twilight         498517 non-null  object 
 12  Nautical_Twilight      498517 non-null  object 
 13  Astronomical_Twilight  498517 non-null  object 
 14  Start_Lat              500000 non-nu

None

In [ ]:
# Drop missing values that are not realistic to impute
preprocessing_df = accidents_df[
    WEATHER_COLUMNS + TEMPORAL_COLUMNS + GEOGRAPHIC_COLUMNS + INFRA_COLUMNS
].dropna(subset=["Sunrise_Sunset", "Timezone", "Visibility(mi)"])

# Convert Start_Time and End_Time to datetime
preprocessing_df["Start_Time"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed")
preprocessing_df["End_Time"] = pd.to_datetime(preprocessing_df["End_Time"], format="mixed")

# State-wise imputation on numeric weather features
# First do date-wise, then falls back to month-wise
preprocessing_df["Date"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed").dt.date
preprocessing_df["Month"] = pd.to_datetime(preprocessing_df["Start_Time"], format="mixed").dt.month
IMPUTED_FEATURES = [
    "Precipitation(in)", "Humidity(%)", "Pressure(in)",
    "Wind_Speed(mph)", "Visibility(mi)", "Temperature(F)"
]
for col in IMPUTED_FEATURES:
    preprocessing_df[col] = (
        preprocessing_df[col]
        .fillna(preprocessing_df.groupby(["State", "Date"])[col].transform("median"))
        .fillna(preprocessing_df.groupby(["State", "Month"])[col].transform("median"))
    )

# drop the remaining NA values
preprocessing_df = preprocessing_df.dropna()
preprocessing_df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 483790 entries, 0 to 499999
Data columns (total 36 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Temperature(F)         483790 non-null  float64       
 1   Humidity(%)            483790 non-null  float64       
 2   Pressure(in)           483790 non-null  float64       
 3   Visibility(mi)         483790 non-null  float64       
 4   Wind_Speed(mph)        483790 non-null  float64       
 5   Precipitation(in)      483790 non-null  float64       
 6   Wind_Direction         483790 non-null  object        
 7   Weather_Condition      483790 non-null  object        
 8   Start_Time             483790 non-null  datetime64[ns]
 9   End_Time               483790 non-null  datetime64[ns]
 10  Sunrise_Sunset         483790 non-null  object        
 11  Civil_Twilight         483790 non-null  object        
 12  Nautical_Twilight      483790 non-null  object   

# Feature Engineering

### Questions we are trying to address: What are the characteristics of the accidents that occur under unusual conditions? How are they associated with higher severity?


In [84]:
# Weather Features
features_df = pd.DataFrame()
WEATHER_FEATURES = [
    "Temperature(F)", "Humidity(%)", "Pressure(in)", "Visibility(mi)",
    "Wind_Speed(mph)", "Precipitation(in)"
]


features_df[WEATHER_FEATURES] = preprocessing_df[WEATHER_FEATURES]

In [85]:
# Temporal features
TEMPORAL_FEATURES = [
    "Start_Time", "End_Time", "Sunrise_Sunset", "Civil_Twilight", "Nautical_Twilight",
    "Astronomical_Twilight"
]
features_df["Hour"] = preprocessing_df["Start_Time"].dt.hour
features_df["Month"] = preprocessing_df["Start_Time"].dt.month
features_df["Year"] = preprocessing_df["Start_Time"].dt.year
features_df["IsRushHour"] = (
    features_df["Hour"].between(7, 9)
    | features_df["Hour"].between(16, 18)
)
features_df

,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Speed(mph),Precipitation(in),Hour,Month,Year,IsRushHour
0,77.0,62.0,29.92,10.0,5.0,0.00,10,6,2019,False
1,45.0,48.0,29.91,10.0,5.0,0.00,23,12,2022,False
2,68.0,73.0,29.79,10.0,13.0,0.00,13,8,2022,False
3,27.0,86.0,28.49,10.0,15.0,0.00,17,2,2022,True
4,42.0,34.0,29.77,10.0,0.0,0.00,1,12,2020,False
...,...,...,...,...,...,...,...,...,...,...
499995,40.0,77.0,29.55,10.0,15.0,0.01,7,12,2021,True
499996,78.0,87.0,29.94,10.0,13.0,0.01,16,12,2021,True
499997,52.0,12.0,24.94,10.0,12.0,0.00,19,4,2022,False
499998,82.0,38.0,28.70,10.0,14.0,0.00,17,5,2020,True


# Feature Selection

### 1. Within-group selection: Identify useful features within each category (weather, time, road infrastructure)

### 2. Between-group selection: Start with the best-performing feature group, then progressively add features from other groups.
